# Código para importar partidas e transformar em CSV

* Site com os box scores (Franca, Pinheiros, Minas, Flamengo, Corinthians)
https://basketball.realgm.com/international/league/59/Brazilian-NBB/team/941/Uniceub-BRB-Brasilia/schedule/2026

In [238]:
# Importação de bibliotecas
import pandas as pd
from bs4 import BeautifulSoup

In [239]:
# Arquivo salvo como html
with open(r"HTML_Partidas\VSPIN_2_L.html", "r", encoding="latin-1") as f:
    conteudo = f.read()

In [240]:
# Criação de parser
soup = BeautifulSoup(conteudo, "html.parser")

In [241]:
# Pega todas as linhas da tabela
linhas = soup.find_all('tr')

In [242]:
# Extrai o texto de cada célula
data = []
for linha in linhas:
    cols = linha.find_all(['td', 'th'])
    cols = [col.get_text(strip=True) for col in cols]
    data.append(cols)

In [243]:
# Exibe a tabela selecionada
df = pd.DataFrame(data[1:])

In [244]:
df[:37]

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,BRA (17-7),17,12,30,5,64,None,None,None,None,None,None,None,None,None,None,None,None
1,PIN (19-4),21,18,18,13,70,None,None,None,None,None,None,None,None,None,None,None,None
2,Advanced,Poss,ORtg,DRtg,None,None,None,None,None,None,None,None,None,None,None,None,None,None
3,BRA,68,94.4,103.2,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4,PIN,68,103.2,94.4,None,None,None,None,None,None,None,None,None,None,None,None,None,None
5,Four Factors,eFG%,TO%,OR%,FTR,None,None,None,None,None,None,None,None,None,None,None,None,None
6,BRA,.391,0.177,0.472,0.145,None,None,None,None,None,None,None,None,None,None,None,None,None
7,PIN,.460,0.251,0.357,0.190,None,None,None,None,None,None,None,None,None,None,None,None,None
8,#,Player,Status,Pos,Min,FGM-A,3PM-A,FTM-A,FIC,Off,Def,Reb,Ast,PF,STL,TO,BLK,PTS
9,2,Kevin Crescenzi,Starter,SG,33:00,6-12,5-8,2-4,13.0,0,4,4,1,1,1,0,0,19


In [245]:
# Extraindo df adversário
df_adv = df[34:36]

In [246]:
df_adv

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
34,,Team,,,,,,,,4,3,7,,,,,,
35,,,,,200,25-63,8-29,12-16,57.0,19,30,49,17,18,9,17,5,70


In [247]:
# Estabelecendo nomes das colunas
cols = ['#','Player','Status','Pos','Min','FGM-A','3PM-A','FTM-A','FIC','Off','Def','Reb','Ast','PF','STL','TO','BLK','PTS']
df_adv.columns = cols

# Trocando de "TEAM" Para o nome da equipe
df_adv = df_adv.replace("Team", "Pinheiros")

# Eliminando colunas desnecessárias
df_adv = df_adv.drop(columns = ['#', 'Status', 'Pos', 'Min', 'FIC'])

# Acertando os dados para iniciar o tratamento final do df
df_adv.loc[35, 'Player'] = 'Pinheiros'
df_adv = df_adv.drop(axis = 1, index = 34)

In [248]:
df_adv

,Player,FGM-A,3PM-A,FTM-A,Off,Def,Reb,Ast,PF,STL,TO,BLK,PTS
35,Pinheiros,25-63,8-29,12-16,19,30,49,17,18,9,17,5,70


In [249]:
# Criando colunas e alterando tipos dos valores
df_adv[['FGM','FGA']] = df_adv['FGM-A'].str.split('-', expand = True)
df_adv[['3PM','3PA']] = df_adv['3PM-A'].str.split('-', expand = True)
df_adv[['FTM','FTA']] = df_adv['FTM-A'].str.split('-', expand = True)
df_adv = df_adv.drop(columns = ['FGM-A','3PM-A','FTM-A'])
df_adv = df_adv.reset_index(drop = True)

if 'Player' in df_adv.columns:
    num_cols = df_adv.columns.drop('Player')
    df_adv[num_cols] = df_adv[num_cols].astype(float)

# Criando colunas calculadas para análise
'''
AST/TO, TS%, ORtg, DRtg, eDiff
'''
df_adv['AST/TO'] = df_adv['Ast']/df_adv['TO']
df_adv['AST/TO'] = df_adv['AST/TO'].round(2)
df_adv['TS%'] = (df_adv['PTS'] * 50)/(df_adv['FGA'] + (0.44 * df_adv['FTA']))
df_adv['TS%'] = df_adv['TS%'].round(2)
df_adv['ORtg'] = (100 * (df_adv['PTS']/((df_adv['FGA'] + (0.44 * df_adv['FTA'])) - df_adv['Off'] + df_adv['TO'])))
df_adv['ORtg'] = df_adv['ORtg'].round(2)
df_adv['DRtg'] = (100 * (64/((69 + (0.44 * 15)) - 19 + 12)))
df_adv['DRtg'] = df_adv['DRtg'].round(2)
df_adv['eDiff'] = df_adv['ORtg'] - df_adv['DRtg']

In [250]:
df_adv

,Player,Off,Def,Reb,Ast,PF,STL,TO,BLK,PTS,...,FGA,3PM,3PA,FTM,FTA,AST/TO,TS%,ORtg,DRtg,eDiff
0,Pinheiros,19.0,30.0,49.0,17.0,18.0,9.0,17.0,5.0,70.0,...,63.0,8.0,29.0,12.0,16.0,1.0,49.97,102.88,93.29,9.59


In [251]:
# Criando um csv, ou anexando a um existente se o adversário ganhou ou perdeu a partida
'''
df_Wadv ou df_Ladv
'''
df_adv.to_csv('Df_Partidas/df_Wadv.csv', index=False, mode='a', header=False)

In [252]:
# Extração do df do Brasília
df_bsb = df[19:21]

In [253]:
df_bsb

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
19,,Team,,,,,,,,2,1,3,,1,,,,
20,,,,,200,21-69,12-42,10-15,46.4,19,20,39,18,18,9,12,2,64


In [254]:
# Estabelecendo nomes das colunas
colsbsb = ['#','Player','Status','Pos','Min','FGM-A','3PM-A','FTM-A','FIC','Off','Def','Reb','Ast','PF','STL','TO','BLK','PTS']
df_bsb.columns = colsbsb

# Trocando de "TEAM" Para o nome da equipe
df_bsb = df_bsb.replace("Team", "Brasília")

# Eliminando colunas desnecessárias
df_bsb = df_bsb.drop(columns = ['#', 'Status', 'Pos', 'Min', 'FIC'])

# Acertando os dados para iniciar o tratamento final do df
df_bsb.loc[20, 'Player'] = 'Brasília'
df_bsb = df_bsb.drop(axis = 1, index = 19)

# Criando colunas e alterando tipos dos valores
df_bsb[['FGM', 'FGA']] = df_bsb['FGM-A'].str.split('-', expand = True)
df_bsb[['3PM', '3PA']] = df_bsb['3PM-A'].str.split('-', expand = True)
df_bsb[['FTM', 'FTA']] = df_bsb['FTM-A'].str.split('-', expand = True)
df_bsb = df_bsb.drop(columns = ['FGM-A', '3PM-A', 'FTM-A'])
df_bsb = df_bsb.reset_index(drop = True)

if 'Player' in df_bsb.columns:
    num_cols = df_bsb.columns.drop('Player')
    df_bsb[num_cols] = df_bsb[num_cols].astype(float)

# Criando colunas calculadas para análise
'''
AST/TO, TS%, ORtg, DRtg, eDiff
'''
df_bsb['AST/TO'] = df_bsb['Ast']/df_bsb['TO']
df_bsb['AST/TO'] = df_bsb['AST/TO'].round(2)
df_bsb['TS%'] = (df_bsb['PTS'] * 50)/(df_bsb['FGA'] + (0.44 * df_bsb['FTA']))
df_bsb['TS%'] = df_bsb['TS%'].round(2)
df_bsb['ORtg'] = df_adv['DRtg']
df_bsb['DRtg'] = df_adv['ORtg']
df_bsb['eDiff'] = df_bsb['ORtg'] - df_bsb['DRtg']

In [255]:
df_bsb

,Player,Off,Def,Reb,Ast,PF,STL,TO,BLK,PTS,...,FGA,3PM,3PA,FTM,FTA,AST/TO,TS%,ORtg,DRtg,eDiff
0,Brasília,19.0,20.0,39.0,18.0,18.0,9.0,12.0,2.0,64.0,...,69.0,12.0,42.0,10.0,15.0,1.5,42.33,93.29,102.88,-9.59


In [256]:
# Criando um csv, ou anexando a um existente se o Brasília ganhou ou perdeu a partida
'''
df_Wbsb ou df_Lbsb
'''
df_bsb.to_csv('Df_Partidas/df_Lbsb.csv', index=False, mode='a', header=False)